In [1]:
import pandas as pd

In [2]:
df = pd.read_csv(r"C:\Users\hp\Desktop\Amazon\CSV Files\amazon_india_2025.csv")

In [3]:
df.shape

(77385, 34)

In [ ]:
df.info()

In [5]:
import numpy as np

df.replace("", np.nan, inplace=True)


In [6]:
dfc = df.copy()

Question 1
Your dataset contains order_date in multiple formats: 'DD/MM/YYYY', 'DD-MM-YY', 'YYYY-MM-DD', and some invalid entries like '32/13/2020'. Clean and standardize all dates to 'YYYY-MM-DD' format, handling invalid dates appropriately.


In [7]:
import pandas as pd

dfc['order_date'] = (
    dfc['order_date']
    .astype('string')
    .str.strip()
    .str.replace(r'[^\d/-]', '', regex=True)
)

dfc['order_date'] = pd.to_datetime(
    dfc['order_date'],
    dayfirst=True,
    errors='coerce'
)

dfc['order_date'] = dfc['order_date'].dt.strftime('%Y-%m-%d')


In [8]:
dfc['order_date'].head(50)

0     2025-08-01
1            NaN
2            NaN
3     2025-04-01
4     2025-03-01
5     2025-07-01
6     2025-09-01
7            NaN
8            NaN
9            NaN
10    2025-10-01
11           NaN
12           NaN
13           NaN
14    2025-01-01
15           NaN
16           NaN
17           NaN
18           NaN
19           NaN
20           NaN
21    2025-10-01
22           NaN
23           NaN
24           NaN
25    2025-01-01
26           NaN
27           NaN
28           NaN
29           NaN
30    2025-03-01
31           NaN
32    2025-01-01
33    2025-05-01
34           NaN
35    2025-01-01
36           NaN
37           NaN
38           NaN
39    2025-06-01
40    2025-07-01
41           NaN
42           NaN
43           NaN
44           NaN
45           NaN
46           NaN
47    2025-08-01
48           NaN
49           NaN
Name: order_date, dtype: object

Question 2
The original_price_inr column contains mixed data types: numeric values, text with '₹' symbols, comma separators ('₹1,25,000'), and some entries like 'Price on Request'. Clean this column to contain only numeric values in Indian Rupees.

In [9]:
dfc['original_price_inr'] = (
    dfc['original_price_inr']
        .astype(str)                      
        .str.replace('₹', '', regex=False) 
        .str.replace(',', '', regex=False)
        .str.replace('Rs ', '', regex=False)
        .str.strip()                
)

dfc['original_price_inr'] = pd.to_numeric(
    dfc['original_price_inr']
)


Question 3
Customer ratings appear in various formats: '5.0', '4 stars', '3/5', '2.5/5.0', and some missing values. Standardize all ratings to numeric scale 1.0-5.0, handling inconsistent formats and missing values strategically.

In [12]:
import pandas as pd
import numpy as np
import re

# Example: if not already loaded
# dfc = pd.read_csv("amazon_india_2015_clean.csv")

def parse_rating(r):
    # 1. Handle missing values
    if pd.isna(r):
        return np.nan

    # 2. If already numeric (int or float), return it
    if isinstance(r, (int, float)):
        return float(r)

    # 3. Convert to string safely
    r = str(r).strip()

    # 4. Handle fraction ratings like "4/5"
    if '/' in r:
        try:
            a, b = r.split('/')
            return (float(a) / float(b)) * 5
        except:
            return np.nan

    # 5. Extract numeric part from strings like:
    #    "4.5 stars", "Rating: 3", "Rated 4 out of 5"
    match = re.search(r'\d+\.?\d*', r)
    if match:
        return float(match.group())

    # 6. Everything else → NaN
    return np.nan


# Apply cleaning
dfc['customer_rating'] = dfc['customer_rating'].apply(parse_rating)

# Optional but recommended: keep ratings between 1 and 5
dfc.loc[
    (dfc['customer_rating'] < 1) | (dfc['customer_rating'] > 5),
    'customer_rating'
] = np.nan

# Round to 2 decimal places
dfc['customer_rating'] = dfc['customer_rating'].round(2)


In [ ]:
dfc['customer_rating'].head(50)

Question 4
The customer_city column has inconsistent naming: 'Bangalore/Bengaluru', 'Mumbai/Bombay', 'Delhi/New Delhi', along with spelling errors and case variations. Standardize all city names and handle geographical variations.

In [15]:
dfc['customer_city'] = (
    dfc['customer_city']
    .str.lower()
    .str.strip()
)
 
city_map = {
    'bangalore': 'Bengaluru',
    'bengaluru': 'Bengaluru',
    'bangalore/bengaluru': 'Bengaluru',
    'bengalore' : 'Bengaluru',
    'Bengaluru' : 'banglore',
    
    'mumbai': 'Mumbai',
    'bombay': 'Mumbai',
    'mumbai/bombay': 'Mumbai',
    'mumba' : 'Mumbai',
    'calcutta' : 'kolkata',

    'delhi': 'Delhi',
    'new delhi': 'Delhi',
    'delhi/new delhi': 'Delhi',
    'delhi NCR' : 'Delhi',
    'delhi ncr' : 'Delhi',

    'chenai' : 'chennai',
    'madras' : 'chennai'
}

dfc['customer_city'] = dfc['customer_city'].replace(city_map)

Question 5
Boolean columns (is_prime_member, is_prime_eligible, is_festival_sale) contain mixed values: True/False, Yes/No, 1/0, Y/N, and some missing entries. Convert all boolean columns to consistent True/False format.


In [16]:
import numpy as np

bool_cols = ['is_prime_member', 'is_prime_eligible', 'is_festival_sale']

for col in bool_cols:
    dfc[col] = dfc[col].replace(['', ' ', 'NA', 'N/A', None, 'None'], np.nan)


bool_map = {
    True: True,
    'True': True,
    'true': True,
    'Yes': True,
    'yes': True,
    'Y': True,
    'y': True,
     1: True,
    
     False: False,
    'False': False,
    'false': False,
    'No': False,
    'no': False,
    'N': False,
    'n': False,
     0: False
}


for col in bool_cols:
    dfc[col] = dfc[col].map(bool_map)

Question 6
Product categories have variations: 'Electronics/Electronic/ELECTRONICS/Electronics & Accessories'. Standardize category names across the dataset and ensure consistent naming conventions.

In [17]:
dfc.columns = dfc.columns.str.strip()

category_map = {
    'electronics': 'Electronics',
    'ELECTRONICS': 'Electronics',
    'electronics & accessories': 'Electronics',
    'Electronicss': 'Electronics',
    'Electronics & Accessories': 'Electronics',
    'Electronic': 'Electronics',
    'clothing': 'Fashion'
}

dfc['category'] = dfc['category'].replace(category_map)

Question 7
The delivery_days column contains negative values, text entries like 'Same Day', '1-2 days', and some unrealistic values like 50 days. Clean this column to contain only valid numeric delivery days.


In [18]:
import numpy as np

days_map = {
    'Express': '0',
    'Same Day': '0',
    '-1': 'None',
    '1-2 days': '2'
}

dfc['delivery_days'] = dfc['delivery_days'].replace(days_map)

Question 8
Identify and handle duplicate transactions where the same customer, product, date, and amount appear multiple times. Some duplicates are genuine (bulk orders) while others are data errors. Develop a strategy to distinguish and handle both cases.


In [19]:
dup_cols = [
    "customer_id",
    "product_id",
    "order_date",
    "final_amount_inr"
]

price_cols = [
    "original_price_inr",
    "discounted_price_inr",
    "subtotal_inr",
    "final_amount_inr"
]

dfc["dup_count"] = (
    dfc.groupby(dup_cols)["transaction_id"]
      .transform("count")
)


dfc["price_identical"] = (
    dfc.groupby(dup_cols)[price_cols]
      .transform("nunique")
      .max(axis=1) == 1
)


dfc["is_high_value"] = dfc["final_amount_inr"] > 5000
dfc["is_bulk_customer"] = dfc["customer_spending_tier"].isin(["Premium"])
dfc["is_bulk_quantity"] = dfc["quantity"] > 1

dfc["is_duplicate_candidate"] = dfc["dup_count"] > 1


In [20]:
df_deduped = dfc[dfc["is_duplicate_candidate"]].drop_duplicates(subset=dup_cols, keep="first")


In [ ]:
print("Rows deleted:", (df_deduped))

In [19]:
len(dfc)

127132

In [21]:
cols_to_drop = [
    "dup_count",
    "price_identical",
    "is_high_value",
    "is_bulk_customer",
    "is_bulk_quantity",
    "is_duplicate_candidate"
]

dfc = dfc.drop(columns=cols_to_drop)

Question 9
The dataset contains outlier prices where some products show prices 100x higher than expected due to data entry errors (decimal point issues). Identify and correct these outliers using statistical methods and domain knowledge.


In [22]:
import numpy as np

dfc["product_median_price"] = (
    dfc.groupby("product_id")["final_amount_inr"]
      .transform("median")
)

dfc["price_outlier"] = (
    dfc["final_amount_inr"] > 50 * dfc["product_median_price"]
)

dfc.loc[dfc["price_outlier"], "final_amount_inr"] /= 100
dfc.loc[dfc["price_outlier"], "discounted_price_inr"] /= 100
dfc.loc[dfc["price_outlier"], "original_price_inr"] /= 100

dfc["subtotal_inr"] = dfc["discounted_price_inr"] * dfc["quantity"]
dfc["final_amount_inr"] = dfc["subtotal_inr"] + dfc["delivery_charges"].fillna(0)

dfc["price_corrected_flag"] = dfc["price_outlier"]

dfc.drop(columns=["product_median_price"], inplace=True)

corrected_rows = dfc[dfc["price_corrected_flag"]]

print(corrected_rows)


Empty DataFrame
Columns: [transaction_id, order_date, customer_id, product_id, product_name, category, subcategory, brand, original_price_inr, discount_percent, discounted_price_inr, quantity, subtotal_inr, delivery_charges, final_amount_inr, customer_city, customer_state, customer_tier, customer_spending_tier, customer_age_group, payment_method, delivery_days, delivery_type, is_prime_member, is_festival_sale, festival_name, customer_rating, return_status, order_month, order_year, order_quarter, product_weight_kg, is_prime_eligible, product_rating, price_outlier, price_corrected_flag]
Index: []

[0 rows x 36 columns]


In [34]:
dfc = dfc.drop(
    columns=[
        "price_outlier",
        "price_corrected_flag"       
    ]
)



Question 10
Payment methods contain inconsistent naming: 'UPI/PhonePe/GooglePay', 'Credit Card/CREDIT_CARD/CC', 'Cash on Delivery/COD/C.O.D'. Standardize payment method categories and create a clean categorical hierarchy.


In [23]:
dfc["payment_method"] = (
    dfc["payment_method"]
    .str.upper()
    .str.replace(".", "", regex=False)
    .str.strip()
)

payment_map = {
    "UPI": "UPI",
    "PHONEPE": "UPI",
    "GOOGLEPAY": "UPI",
    "GPAY": "UPI",
    "PAYTM": "UPI",

    "CREDIT CARD": "Credit Card",
    "CREDIT_CARD": "Credit Card",
    "CC": "Credit Card",

    "DEBIT CARD": "Debit Card",
    "DC": "Debit Card",

    "COD": "Cash on Delivery",
    "CASH ON DELIVERY": "Cash on Delivery",

    "NET BANKING": "Net Banking"
}

dfc["payment_method"] = dfc["payment_method"].replace(payment_map)

In [25]:
dfc['original_price_inr'] = (
    dfc['original_price_inr']
    .astype(str)
    .str.replace('-', '', regex=False)
    .astype(float)
)


In [26]:
dfc["payment_method"].unique()

array(['UPI', 'Credit Card', 'BNPL', 'Cash on Delivery', 'Debit Card',
       'Net Banking', 'WALLET'], dtype=object)

In [27]:
import pandas as pd

columns = [
    "transaction_id","order_date","customer_id","product_id","product_name",
    "category","subcategory","brand","original_price_inr","discount_percent",
    "discounted_price_inr","quantity","subtotal_inr"
]
unique_values = {}

for col in columns:
    if col in dfc.columns:
        unique_values[col] = sorted(dfc[col].dropna().astype(str).unique())
    else:
        unique_values[col] = []

print("Unique values for each column:\n")
for col, values in unique_values.items():
    print(f"{col}: {values}\n")


Unique values for each column:

transaction_id: ['TXN_2025_00000001', 'TXN_2025_00000002', 'TXN_2025_00000003', 'TXN_2025_00000004', 'TXN_2025_00000005', 'TXN_2025_00000006', 'TXN_2025_00000007', 'TXN_2025_00000008', 'TXN_2025_00000009', 'TXN_2025_00000010', 'TXN_2025_00000011', 'TXN_2025_00000012', 'TXN_2025_00000013', 'TXN_2025_00000014', 'TXN_2025_00000015', 'TXN_2025_00000016', 'TXN_2025_00000017', 'TXN_2025_00000018', 'TXN_2025_00000019', 'TXN_2025_00000020', 'TXN_2025_00000021', 'TXN_2025_00000022', 'TXN_2025_00000023', 'TXN_2025_00000024', 'TXN_2025_00000025', 'TXN_2025_00000026', 'TXN_2025_00000027', 'TXN_2025_00000028', 'TXN_2025_00000029', 'TXN_2025_00000030', 'TXN_2025_00000031', 'TXN_2025_00000032', 'TXN_2025_00000033', 'TXN_2025_00000034', 'TXN_2025_00000035', 'TXN_2025_00000036', 'TXN_2025_00000037', 'TXN_2025_00000038', 'TXN_2025_00000039', 'TXN_2025_00000040', 'TXN_2025_00000041', 'TXN_2025_00000042', 'TXN_2025_00000043', 'TXN_2025_00000044', 'TXN_2025_00000045', 'TXN_2

In [28]:
import pandas as pd

columns = [
    "delivery_charges",
    "final_amount_inr","customer_city","customer_state","customer_tier",
    "customer_spending_tier","customer_age_group","payment_method","delivery_days",
    "delivery_type","is_prime_member","is_festival_sale",
    "festival_name"
]
unique_values = {}

for col in columns:
    if col in dfc.columns:
        unique_values[col] = sorted(dfc[col].dropna().astype(str).unique())
    else:
        unique_values[col] = []

print("Unique values for each column:\n")
for col, values in unique_values.items():
    print(f"{col}: {values}\n")


Unique values for each column:

delivery_charges: ['0.0', '40.0']

final_amount_inr: ['1000.93', '10000.47', '10000.68', '100007.61000000002', '10001.08', '100013.73000000001', '10002.24', '100031.88', '100040.45999999999', '10005.17', '10005.58', '100050.0', '100057.92', '100058.99', '10006.94', '100069.21', '100075.78', '10008.07', '10008.56', '10010.25', '10010.83', '10011.02', '100119.69', '10012.22', '10012.77', '10012.93', '100124.28', '100126.55', '100128.26999999999', '10013.03', '10013.47', '10014.85', '10015.3', '10016.2', '10016.74', '10016.93', '10017.11', '10017.21', '100179.29', '10018.87', '100186.46', '10019.82', '1002.32', '10020.93', '10021.2', '10021.86', '100213.53', '10022.47', '10022.82', '10024.55', '10024.65', '10024.75', '10026.29', '10027.35', '100277.47', '100278.96', '10028.42', '10029.3', '100291.59', '100319.07', '100327.16', '100329.76', '10033.65', '100333.2', '100335.18', '100340.2', '10035.32', '100353.4', '100359.46', '10036.25', '10036.49', '100368.4

In [29]:
import pandas as pd

columns = [    "customer_rating","return_status","order_month","order_year","order_quarter",
    "product_weight_kg","is_prime_eligible","product_rating"
]
unique_values = {}

for col in columns:
    if col in dfc.columns:
        unique_values[col] = sorted(dfc[col].dropna().astype(str).unique())
    else:
        unique_values[col] = []

print("Unique values for each column:\n")
for col, values in unique_values.items():
    print(f"{col}: {values}\n")


Unique values for each column:

customer_rating: ['3.0', '3.5', '4.0', '4.5', '5.0']

return_status: ['Cancelled', 'Delivered', 'Returned']

order_month: ['1', '10', '11', '12', '2', '3', '4', '5', '6', '7', '8', '9']

order_year: ['2025']

order_quarter: ['1', '2', '3', '4']

product_weight_kg: ['0.03', '0.04', '0.05', '0.06', '0.07', '0.08', '0.09', '0.1', '0.11', '0.12', '0.13', '0.14', '0.15', '0.16', '0.17', '0.18', '0.19', '0.2', '0.21', '0.22', '0.23', '0.24', '0.25', '0.26', '0.27', '0.28', '0.29', '0.3', '0.31', '0.32', '0.33', '0.34', '0.35', '0.36', '0.37', '0.38', '0.39', '0.4', '0.41', '0.42', '0.43', '0.44', '0.45', '0.46', '0.47', '0.48', '0.49', '0.5', '0.51', '0.52', '0.53', '0.54', '0.55', '0.56', '0.57', '0.58', '0.59', '0.6', '0.61', '0.62', '0.63', '0.64', '0.65', '0.66', '0.67', '0.68', '0.69', '0.7', '0.71', '0.72', '0.73', '0.75', '0.76', '0.77', '0.78', '0.79', '1.2', '1.21', '1.24', '1.27', '1.29', '1.31', '1.32', '1.33', '1.36', '1.37', '1.38', '1.39', '1.4',

In [30]:
print(len(dfc.columns))
print(dfc.columns.tolist())


36
['transaction_id', 'order_date', 'customer_id', 'product_id', 'product_name', 'category', 'subcategory', 'brand', 'original_price_inr', 'discount_percent', 'discounted_price_inr', 'quantity', 'subtotal_inr', 'delivery_charges', 'final_amount_inr', 'customer_city', 'customer_state', 'customer_tier', 'customer_spending_tier', 'customer_age_group', 'payment_method', 'delivery_days', 'delivery_type', 'is_prime_member', 'is_festival_sale', 'festival_name', 'customer_rating', 'return_status', 'order_month', 'order_year', 'order_quarter', 'product_weight_kg', 'is_prime_eligible', 'product_rating', 'price_outlier', 'price_corrected_flag']


In [31]:
dfc[dfc.duplicated()]

,transaction_id,order_date,customer_id,product_id,product_name,category,subcategory,brand,original_price_inr,discount_percent,...,customer_rating,return_status,order_month,order_year,order_quarter,product_weight_kg,is_prime_eligible,product_rating,price_outlier,price_corrected_flag


In [32]:
import pandas as pd

decimal_cols = dfc.select_dtypes(include=['float', 'float64']).columns

dfc[decimal_cols] = dfc[decimal_cols].round(2)
print(decimal_cols)


Index(['original_price_inr', 'discount_percent', 'discounted_price_inr',
       'subtotal_inr', 'delivery_charges', 'final_amount_inr',
       'customer_rating', 'product_weight_kg', 'product_rating'],
      dtype='object')


In [36]:
dfc.to_csv(r"C:\Users\hp\Desktop\Amazon\CSV_Clean_Files\amazon_india_2025_clean.csv",header='infer',index=False)

In [35]:
dfc

,transaction_id,order_date,customer_id,product_id,product_name,category,subcategory,brand,original_price_inr,discount_percent,...,is_festival_sale,festival_name,customer_rating,return_status,order_month,order_year,order_quarter,product_weight_kg,is_prime_eligible,product_rating
0,TXN_2025_00000001,2025-08-01,CUST_2025_00005600,PROD_000627,Oppo F11 Pro 128GB Black,Electronics,Smartphones,Oppo,10234.12,0.00,...,False,NaN,4.5,Delivered,1,2025,1,0.15,True,4.4
1,TXN_2025_00000002,NaN,CUST_2022_00027099,PROD_001699,Samsung Slate 4GB RAM Silver,Electronics,Tablets,Samsung,38241.08,0.00,...,False,NaN,NaN,Returned,1,2025,1,0.64,NaN,3.4
2,TXN_2025_00000003,NaN,CUST_2021_00027917,PROD_001242,Apple iPhone 16 Plus 64GB Black,Electronics,Smartphones,Apple,121974.26,32.04,...,True,Republic Day Sale,NaN,Returned,1,2025,1,0.18,True,3.4
3,TXN_2025_00000004,2025-04-01,CUST_2025_00004184,PROD_000979,Samsung Galaxy S22+ 128GB White,Electronics,Smartphones,Samsung,59075.70,0.00,...,False,NaN,3.5,Delivered,1,2025,1,0.24,False,3.3
4,TXN_2025_00000005,2025-03-01,CUST_2025_00005205,PROD_001876,Apple Watch Premium,Electronics,Smart Watch,Apple,74269.31,0.00,...,False,NaN,5.0,Returned,1,2025,1,0.05,NaN,4.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
77380,TXN_2025_00019019_DUP,2025-07-04,CUST_2022_00004500,PROD_000136,Samsung Galaxy S7 Edge 16GB Blue,Electronics,Smartphones,Samsung,67658.04,0.00,...,False,NaN,3.5,Delivered,4,2025,2,0.16,True,3.4
77381,TXN_2025_00053859_DUP,NaN,CUST_2025_00013666,PROD_000527,Samsung Galaxy A50 64GB White,Electronics,Smartphones,Samsung,19905.88,24.29,...,False,NaN,3.5,Delivered,10,2025,4,0.20,False,4.2
77382,TXN_2025_00030006_DUP,NaN,CUST_2025_00011941,PROD_000839,Samsung Galaxy Note 21 256GB White,Electronics,Smartphones,Samsung,75655.18,30.26,...,True,Back to School,4.0,Delivered,6,2025,2,0.22,True,4.1
77383,TXN_2025_00066716_DUP,NaN,CUST_2025_00000199,PROD_001581,ASUS Gaming 4GB RAM Black,Electronics,Laptops,ASUS,125461.24,0.00,...,False,NaN,4.5,Delivered,11,2025,4,2.57,False,3.0
